

**The Hybrid Equation: KNOWN PHYSICS + UDE**
$$ C_m \frac{dV}{dt} = I_{ext} - \mathbf{NN(V)} - I_{K}(V) - I_{L}(V) $$


In [2]:
using Lux, SciMLSensitivity, DifferentialEquations, Zygote, Optimisers
using Optimization, OptimizationOptimisers, ComponentArrays, Random, Statistics, Printf, JLD2

Random.seed!(1234)

TaskLocalRNG()

In [3]:
# Load and cast to Float32 for speed
@load "Data/synthetic_data/noise_0_hh_2d_model.jld" V 
V = Float32.(V) 
const V_mean, V_std = mean(V), std(V)
norm_input(v) = (v - V_mean) / V_std

norm_input (generic function with 1 method)

In [4]:
const Cm = 1.0f0
const g_Na = 120.0f0
const E_Na = 50.0f0
const g_L = 0.3f0
const E_L = -54.387f0
const g_K = 36.0f0
const E_K = -77.0f0

# Rate functions using Float32 literals
alpha_n(V) = 0.01f0 * (V + 55.0f0) / (1.0f0 - exp(-(V + 55.0f0) / 10.0f0))
beta_n(V)  = 0.125f0 * exp(-(V + 65.0f0) / 80.0f0)
alpha_m(V) = 0.1f0 * (V + 40.0f0) / (1.0f0 - exp(-(V + 40.0f0) / 10.0f0))
beta_m(V)  = 4.0f0 * exp(-(V + 65.0f0) / 18.0f0)
alpha_h(V) = 0.07f0 * exp(-(V + 65.0f0) / 20.0f0)
beta_h(V)  = 1.0f0 / (1.0f0 + exp(-(V + 35.0f0) / 10.0f0))

# Dynamics helpers
n_inf(V) = alpha_n(V) / (alpha_n(V) + beta_n(V))
tau_n(V) = 1.0f0 / (alpha_n(V) + beta_n(V))

tau_n (generic function with 1 method)

In [5]:
# Architecture: 1->32, 32->16, 16->1
nn = Lux.Chain(
    Lux.Dense(1 => 32, tanh),
    Lux.Dense(32 => 16, tanh),
    Lux.Dense(16 => 1)
)
# Initialize parameters and state
rng = Random.default_rng()
p, st = Lux.setup(rng, nn)
p = ComponentArray(p) |> f32  # Parameters as a vector for the optimizer

ComponentVector{Float32}(layer_1 = (weight = Float32[-1.9322631; 0.46108413; … ; -0.33483955; 2.1393096;;], bias = Float32[0.29686153, 0.0059046745, 0.85494876, 0.035732627, -0.5419055, 0.4994383, 0.5092691, 0.10506892, -0.42743325, -0.83311987  …  -0.30362916, 0.06095326, -0.9308648, 0.829281, 0.47381628, -0.33287466, -0.029481769, -0.3040092, 0.42658257, 0.057388067]), layer_2 = (weight = Float32[0.39240673 0.46854094 … -0.041896254 0.25573024; -0.35136715 -0.4161375 … -0.41900465 -0.19239776; … ; -0.29306573 -0.22820212 … -0.1253588 -0.4433559; -0.24962813 0.13448739 … 0.016990984 -0.44502175], bias = Float32[-0.09335567, 0.16351484, -0.074290715, -0.14872599, -0.084965244, 0.15247276, -0.10258958, 0.0064060893, 0.13494799, -0.01309178, 0.056955796, 0.17298207, -0.057211228, -0.0773036, 0.050865263, -0.12558775]), layer_3 = (weight = Float32[0.17893417 -0.21077445 … 0.23041853 -0.07263824], bias = Float32[-0.12491843]))

In [6]:


function Stimulus(t)
    if (t >= 10.0 && t < 11.0) || (t >= 30.0 && t < 31.0)
        return 20.0
    else
        return 0.0
    end
end

Stimulus (generic function with 1 method)

In [7]:
function hodgkin_huxley_UDE!(du, u, p, t)
    V_curr, n = u
    # Lux requires the state (st) to be passed, but we don't update it here
    pred_I_Na, _ = nn([norm_input(V_curr)], p, st)
    
    I_ext = Stimulus(t) # Defined in previous steps
    I_K = 36.0f0 * n^4 * (V_curr + 77.0f0)
    I_L = 0.3f0 * (V_curr + 54.387f0)   
    
    du[1] = (I_ext - pred_I_Na[1] - I_K - I_L) / 1.0f0
    du[2] = (n_inf(V_curr) - n) / tau_n(V_curr)
end

u0 = Float32[-65.0, 0.31767]
tspan = (0.0f0, 50.0f0)
t_train = range(0.0f0, 50.0f0, length=length(V))
prob = ODEProblem(hodgkin_huxley_UDE!, u0, tspan, p)

ODEProblem with uType Vector{Float32} and tType Float32. In-place: true
Non-trivial mass matrix: false
timespan: (0.0f0, 50.0f0)
u0: 2-element Vector{Float32}:
 -65.0
   0.31767

In [8]:
function predict_ude(p)
    _prob = remake(prob, p=p)
    
    solve(_prob, TRBDF2(), saveat=t_train, 
          reltol=1f-3, abstol=1f-4, 
          dtmin=1f-7, dtmax=1f-1,
          sensealg=InterpolatingAdjoint(autojacvec=ZygoteVJP()))
end

predict_ude (generic function with 1 method)

In [12]:
function loss_function(p, _)
    sol = predict_ude(p)
    if sol.retcode != :Success
        return 1e6f0 
    end
    
    l = sum(abs2, sol[1,:] .- V)
    return l 
end


loss_function (generic function with 1 method)

In [ ]:
const SAVE_PATH = "UDE_2D_HH_best_params.jld2"

"UDE_2D_HH_best_params.jld2"

In [9]:
load("UDE_2D_HH_best_params.jld2")

Dict{String, Any} with 3 entries:
  "best_loss"    => 37.0809
  "params"       => (layer_1 = (weight = Float32[-0.8171; 0.710446; … ; -1.3650…
  "loss_history" => Float32[26696.7, 26138.7, 25581.7, 25034.1, 24505.6, 24006.…

In [ ]:
 # Define a path for saving
losses = Float32[]
min_loss = Inf32

Inf32

In [10]:
loaded_checkpoint = load(SAVE_PATH) # Assuming load returns the dictionary
p = loaded_checkpoint["params"]
global losses = loaded_checkpoint["loss_history"]
global min_loss = loaded_checkpoint["best_loss"]
function callback(state, l) 
    push!(losses, l)
    iter = length(losses)

    if l < min_loss
        global min_loss = l
        jldsave(SAVE_PATH; params=state.u, loss_history=losses, best_loss=min_loss)
    end

    # Simplified using Julia's string interpolation ($)
    println("Iter $iter | Loss $l | Best $min_loss")

    return false
end

callback (generic function with 1 method)

In [13]:
 # Initialize with a very large number
using OptimizationOptimJL  # Required for LBFGS/BFGS
optf = OptimizationFunction(loss_function, Optimization.AutoZygote())
optprob = OptimizationProblem(optf, p)



OptimizationProblem. In-place: true
u0: ComponentVector{Float32}(layer_1 = (weight = Float32[-0.8171002; 0.7104459; … ; -1.3650937; 2.8812716;;], bias = Float32[1.6105343, -0.88714087, 1.2648202, -0.67065376, -1.6672946, -0.2827297, 1.9598283, 1.165802, 0.10822328, -0.8557887  …  -0.47375348, -0.5197562, -1.1785907, 0.6852939, 1.3502817, 0.49394602, 0.056820177, -2.2628622, 1.0455846, -0.9206277]), layer_2 = (weight = Float32[0.01335258 1.069846 … -0.7248409 2.5229506; -0.32842493 -0.00064232055 … -0.30546385 -0.25035673; … ; 0.50747013 -0.1653899 … 0.57516235 -1.0887295; -1.1761044 0.9179643 … -0.7014502 0.3856951], bias = Float32[-0.028628983, 1.148682, -0.8217288, -0.07975421, 0.4147325, 1.6942332, 0.72392607, 1.0142996, 0.89219946, -0.66713846, 1.4210894, -1.3192666, -0.10551323, -0.36398208, 1.8215286, -0.6340272]), layer_3 = (weight = Float32[-4.020828 -4.0478745 … -3.4078088 -5.8270955], bias = Float32[-3.6754603]))

In [16]:
println("--- Starting Stage 1: Adam 0.05 ---")
res1 = solve(optprob, OptimizationOptimisers.Adam(0.05f0), callback=callback, maxiters=1)


--- Starting Stage 1: Adam 0.05 ---
Iter 102 | Loss 37.080753 | Best 37.080753
Iter 103 | Loss 37.080753 | Best 37.080753


retcode: Default
u: ComponentVector{Float32}(layer_1 = (weight = Float32[-0.8171002; 0.7104459; … ; -1.3650937; 2.8812716;;], bias = Float32[1.6105343, -0.88714087, 1.2648202, -0.67065376, -1.6672946, -0.2827297, 1.9598283, 1.165802, 0.10822328, -0.8557887  …  -0.47375348, -0.5197562, -1.1785907, 0.6852939, 1.3502817, 0.49394602, 0.056820177, -2.2628622, 1.0455846, -0.9206277]), layer_2 = (weight = Float32[0.01335258 1.069846 … -0.7248409 2.5229506; -0.32842493 -0.00064232055 … -0.30546385 -0.25035673; … ; 0.50747013 -0.1653899 … 0.57516235 -1.0887295; -1.1761044 0.9179643 … -0.7014502 0.3856951], bias = Float32[-0.028628983, 1.148682, -0.8217288, -0.07975421, 0.4147325, 1.6942332, 0.72392607, 1.0142996, 0.89219946, -0.66713846, 1.4210894, -1.3192666, -0.10551323, -0.36398208, 1.8215286, -0.6340272]), layer_3 = (weight = Float32[-4.020828 -4.0478745 … -3.4078088 -5.8270955], bias = Float32[-3.6754603]))

In [17]:
# --- STAGE 1: Adam (High Learning Rate) ---
println("--- Starting Stage 2: Adam 0.05 ---")
optprob2 = remake(optprob, u0 = res1.u)
res2 = solve(optprob2, OptimizationOptimisers.Adam(0.01f0), callback=callback, maxiters=1)


--- Starting Stage 2: Adam 0.05 ---
Iter 104 | Loss 37.080753 | Best 37.080753
Iter 105 | Loss 37.080753 | Best 37.080753


retcode: Default
u: ComponentVector{Float32}(layer_1 = (weight = Float32[-0.8171002; 0.7104459; … ; -1.3650937; 2.8812716;;], bias = Float32[1.6105343, -0.88714087, 1.2648202, -0.67065376, -1.6672946, -0.2827297, 1.9598283, 1.165802, 0.10822328, -0.8557887  …  -0.47375348, -0.5197562, -1.1785907, 0.6852939, 1.3502817, 0.49394602, 0.056820177, -2.2628622, 1.0455846, -0.9206277]), layer_2 = (weight = Float32[0.01335258 1.069846 … -0.7248409 2.5229506; -0.32842493 -0.00064232055 … -0.30546385 -0.25035673; … ; 0.50747013 -0.1653899 … 0.57516235 -1.0887295; -1.1761044 0.9179643 … -0.7014502 0.3856951], bias = Float32[-0.028628983, 1.148682, -0.8217288, -0.07975421, 0.4147325, 1.6942332, 0.72392607, 1.0142996, 0.89219946, -0.66713846, 1.4210894, -1.3192666, -0.10551323, -0.36398208, 1.8215286, -0.6340272]), layer_3 = (weight = Float32[-4.020828 -4.0478745 … -3.4078088 -5.8270955], bias = Float32[-3.6754603]))

In [18]:
# --- STAGE 2: Adam (Fine-Tuning) ---
# Purpose: Smooth out the membrane potential oscillations
println("--- Starting Stage 3: Adam 0.005 ---")
optprob3 = remake(optprob, u0 = res2.u)
res3 = solve(
    optprob3, 
    OptimizationOptimisers.Adam(0.005f0), # Fully qualified name
    callback=callback, 
    maxiters=250
)

--- Starting Stage 3: Adam 0.005 ---
Iter 106 | Loss 37.080753 | Best 37.080753
Iter 107 | Loss 227.75381 | Best 37.080753
Iter 108 | Loss 68.25406 | Best 37.080753
Iter 109 | Loss 46.099384 | Best 37.080753
Iter 110 | Loss 52.313766 | Best 37.080753
Iter 111 | Loss 112.840904 | Best 37.080753
Iter 112 | Loss 60.244564 | Best 37.080753
Iter 113 | Loss 53.468956 | Best 37.080753
Iter 114 | Loss 48.137154 | Best 37.080753
Iter 115 | Loss 61.29133 | Best 37.080753
Iter 116 | Loss 61.479683 | Best 37.080753
Iter 117 | Loss 132.35483 | Best 37.080753
Iter 118 | Loss 42.66036 | Best 37.080753
Iter 119 | Loss 60.100327 | Best 37.080753
Iter 120 | Loss 52.735954 | Best 37.080753
Iter 121 | Loss 51.548367 | Best 37.080753
Iter 122 | Loss 52.87071 | Best 37.080753
Iter 123 | Loss 52.591892 | Best 37.080753
Iter 124 | Loss 52.461334 | Best 37.080753
Iter 125 | Loss 55.062763 | Best 37.080753
Iter 126 | Loss 39.525578 | Best 37.080753
Iter 127 | Loss 40.3953 | Best 37.080753
Iter 128 | Loss 40.025

retcode: Default
u: ComponentVector{Float32}(layer_1 = (weight = Float32[-0.79367083; 0.7039552; … ; -1.3770392; 2.8299615;;], bias = Float32[1.6162598, -0.88771176, 1.2698886, -0.6811071, -1.6524132, -0.37523133, 1.975712, 1.2335687, 0.10524881, -0.85866493  …  -0.46653864, -0.5354247, -1.2035642, 0.6676378, 1.3245399, 0.56533736, 0.08573091, -2.2641737, 1.0383424, -0.9900934]), layer_2 = (weight = Float32[-0.059517607 1.1412159 … -0.7945547 2.371696; -0.36669898 0.03777773 … -0.3440488 -0.21177076; … ; 0.717949 -0.3550236 … 0.7483504 -1.2615998; -1.1436456 0.9417227 … -0.7282375 0.4122056], bias = Float32[-0.10196092, 1.1872678, -0.8382535, -0.061041966, 0.4233504, 1.7386105, 0.7199214, 1.0220335, 0.931655, -0.6739583, 1.4210894, -1.3349822, -0.14070341, -0.36456665, 1.6486583, -0.6075273]), layer_3 = (weight = Float32[-4.099362 -4.105345 … -3.4665918 -5.8846908], bias = Float32[-3.732926]))

In [ ]:
# --- STAGE 2: Adam (Fine-Tuning) ---
# Purpose: Smooth out the membrane potential oscillations
println("--- Starting Stage 4: Adam 0.0025 ---")
optprob6 = remake(optprob, u0 = res5.u)
res6 = solve(
    optprob6, 
    OptimizationOptimisers.Adam(0.00025f0), # Fully qualified name
    callback=callback, 
    maxiters=500
)

--- Starting Stage 4: Adam 0.0025 ---
Iter 866 | Loss 18.433968 | Best 18.433968
Iter 867 | Loss 47.340515 | Best 18.433968
Iter 868 | Loss 20.876238 | Best 18.433968
Iter 869 | Loss 70.95482 | Best 18.433968
Iter 870 | Loss 18.8603 | Best 18.433968
Iter 871 | Loss 20.927279 | Best 18.433968
Iter 872 | Loss 20.969473 | Best 18.433968
Iter 873 | Loss 20.534246 | Best 18.433968
Iter 874 | Loss 19.51177 | Best 18.433968
Iter 875 | Loss 19.287878 | Best 18.433968
Iter 876 | Loss 19.193562 | Best 18.433968
Iter 877 | Loss 19.36845 | Best 18.433968
Iter 878 | Loss 23.298468 | Best 18.433968
Iter 879 | Loss 26.216017 | Best 18.433968
Iter 880 | Loss 33.37123 | Best 18.433968
Iter 881 | Loss 25.591478 | Best 18.433968
Iter 882 | Loss 116.38527 | Best 18.433968
Iter 883 | Loss 23.90994 | Best 18.433968
Iter 884 | Loss 24.064205 | Best 18.433968
Iter 885 | Loss 59.082195 | Best 18.433968
Iter 886 | Loss 61.18021 | Best 18.433968
Iter 887 | Loss 37.317635 | Best 18.433968


In [20]:
# --- STAGE 4: L-BFGS (Memory-Efficient Refinement) ---
# Purpose: Faster convergence for high-dimensional parameter spaces (609 params)
println("--- Starting Stage 4: L-BFGS ---")

# Remake the problem using the weights from the previous stage (e.g., res2 or res3)
optprob5 = remake(optprob, u0 = res4.u) 

res5 = solve(
    optprob5, 
    OptimizationOptimJL.LBFGS(), # Fully qualified name
    callback = callback, 
    maxiters = 1000              # L-BFGS can handle more iterations efficiently
)
println("All Optimization Stages Complete.")
println("Final Best Loss: ", min_loss)
# Pass these results to your final BFGS stage
# optprob5 = remake(optprob, u0 = res4.u)

--- Starting Stage 4: L-BFGS ---
Iter 858 | Loss 19.81578 | Best 19.81567
Iter 859 | Loss 19.34665 | Best 19.34665
Iter 860 | Loss 19.344519 | Best 19.344519
Iter 861 | Loss 19.019596 | Best 19.019596
Iter 862 | Loss 18.434586 | Best 18.434586
Iter 863 | Loss 18.434109 | Best 18.434109
Iter 864 | Loss 18.43401 | Best 18.43401
Iter 865 | Loss 18.43401 | Best 18.43401
All Optimization Stages Complete.
Final Best Loss: 18.43401
